# Machine Learning in Python - Project 

Due Friday, Apr 10th by 4 pm.

B296110 Ming Cen, B300238 Darren Lim

## Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Introduction

The exploratory data analysis and feature engineering phase aims to transform the raw data from the UNICEF Malawi survey into a structured feature set suitable for machine learning. Since the dataset contains numerical variables, ordinal categorical variables, and nominal categorical variables collected from questionnaires for children, mothers, and households, the primary task is to understand the data structure and determine the handling method for each variable. This includes examining the target distribution, identifying missing and non-substantive responses, checking variable types, and reviewing the category composition of key categorical features. These steps are not only used to improve data quality but also to ensure that subsequent preprocessing decisions are substantively meaningful.

Feature engineering focuses on making variables more consistent, interpretable, and suitable for modeling. Non-substantive answers were recoded as missing values, categorical variables stored in string form were converted to numerical form, and variables with too many missing values were deleted. Categorical variables were divided into two groups: ordinal variables and nominal variables, for appropriate processing. For ordinal variables, rare or overlapping categories were merged, and their natural ordering structure was retained. For nominal variables, categories with similar semantics were grouped together to reduce sparsity, while preserving the broader social significance of the original survey items (such as water source, sanitation facilities, housing quality, and ethnicity).

# Exploratory Data Analysis and Feature Engineering

In [2]:
# 1. load data and build target
df = pd.read_csv("unicef_malawi.csv")
df_model = df.copy()
df_model = df_model[df_model["FCF26"].notna() & (df_model["FCF26"] != "NO RESPONSE")].copy()
df_model["target"] = (df_model["FCF26"] != "NEVER").astype(int)
X = df_model.drop(columns=["FCF26", "target", "HH1", "HH2", "LN", "FS4"]).copy()
y = df_model["target"].copy()
print("data shape:", df.shape, "; X shape:", X.shape, "; y shape:", y.shape)
# 2. split data
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=11205, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=11205, stratify=y_temp)
print("X_train:", X_train.shape, "; X_val:", X_val.shape, "; X_test:", X_test.shape,"; y_train:", y_train.shape, "; y_val:", y_val.shape,"; y_test:", y_test.shape)

data shape: (13162, 87) ; X shape: (13036, 82) ; y shape: (13036,)
X_train: (9125, 82) ; X_val: (1955, 82) ; X_test: (1956, 82) ; y_train: (9125,) ; y_val: (1955,) ; y_test: (1956,)


When examining the raw data, I found that "CL13", "MA2", "LS2", and "WS4" are all numerical variables. Moreover, there are too many meaningless values in the raw data, which are no different from NaN.

In [3]:
# 3. define column types from training data only
num_cols_before = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols_before = X_train.select_dtypes(exclude=np.number).columns.tolist()
print("before conversion -> num_cols:", len(num_cols_before), "; cat_cols:", len(cat_cols_before))
move_to_num = ["CL13", "MA2", "LS2", "WS4"]
for col in move_to_num:
    if col in X_train.columns:
        X_train[col] = pd.to_numeric(X_train[col], errors="coerce")
        X_val[col] = pd.to_numeric(X_val[col], errors="coerce")
        X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()
print("after conversion  -> num_cols:", len(num_cols), "; cat_cols:", len(cat_cols))

before conversion -> num_cols: 6 ; cat_cols: 76
after conversion  -> num_cols: 10 ; cat_cols: 72


In [4]:
# 4. remove  non-substantive strings appear in a dataset
non_substantive_values = ["NO RESPONSE", "DK", "DON'T KNOW", "DONT KNOW", "DON’T KNOW","DK / NO OPINION", "MISSING", "NOT ASKED", "REFUSED", "NO ANSWER", "UNKNOWN"] # Possible representations of non-substantive strings
for name, df_part in [("X_train", X_train), ("X_val", X_val), ("X_test", X_test)]:
    before_counts = {}
    for v in non_substantive_values:
        before_counts[v] = (df_part == v).sum().sum()
    before_total = sum(before_counts.values())
    df_part.replace(non_substantive_values, np.nan, inplace=True)
    after_counts = {}
    for v in non_substantive_values:
        after_counts[v] = (df_part == v).sum().sum()
    after_total = sum(after_counts.values())

Due to the presence of numerous missing values, I decided to remove features and samples with excessively high missing rates based on the literature. Specifically, features with a missing rate > 40% and samples with a missing rate > 20% were removed [1-2].

[1] Madley-Dowd, P., Hughes, R., Tilling, K., & Heron, J. (2019). The proportion of missing data should not be used to guide decisions on multiple imputation. Journal of Clinical Epidemiology, 110, 63–73. https://doi.org/10.1016/j.jclinepi.2019.02.016

[2] Marino, M., Lucas, J., Latour, E., & Heintzman, J. D. (2021). Missing data in primary care research: Importance, implications and approaches. Family Practice, 38(2), 199–202. https://doi.org/10.1093/fampra/cmaa134

In [5]:
# 5. remove features and samples with too many NaN values
feature_missing_threshold = 0.40
sample_missing_threshold = 0.20
n_features_before = X_train.shape[1]
n_train_before = X_train.shape[0]
n_val_before = X_val.shape[0]
n_test_before = X_test.shape[0]
feature_missing_rate = X_train.isna().mean()
keep_cols = feature_missing_rate[feature_missing_rate <= feature_missing_threshold].index
X_train = X_train[keep_cols].copy()
X_val = X_val[keep_cols].copy()
X_test = X_test[keep_cols].copy()
n_features_after = X_train.shape[1]
train_keep = X_train.isna().mean(axis=1) <= sample_missing_threshold
val_keep = X_val.isna().mean(axis=1) <= sample_missing_threshold
test_keep = X_test.isna().mean(axis=1) <= sample_missing_threshold
X_train = X_train.loc[train_keep].copy()
y_train = y_train.loc[train_keep].copy()
X_val = X_val.loc[val_keep].copy()
y_val = y_val.loc[val_keep].copy()
X_test = X_test.loc[test_keep].copy()
y_test = y_test.loc[test_keep].copy()
n_train_after = X_train.shape[0]
n_val_after = X_val.shape[0]
n_test_after = X_test.shape[0]
print("features before:", n_features_before, "; features after:", n_features_after)
print("train samples before:", n_train_before,"; train samples after:", n_train_after)
print("val samples before:", n_val_before, "; val samples after:", n_val_after)
print("test samples before:", n_test_before, "; test samples after:", n_test_after)

features before: 82 ; features after: 81
train samples before: 9125 ; train samples after: 8860
val samples before: 1955 ; val samples after: 1880
test samples before: 1956 ; test samples after: 1879


In [6]:
# 6. clean and merge ordinal categorical variables
ordinal_candidates = ["CB5A", "CB5B","WB6A", "WB6B","WB14","AF10", "AF11", "AF12","LS1", "LS3", "LS4","VT20", "VT21"]
ordinal_cols = [col for col in ordinal_candidates if col in X_train.columns]
def clean_merge_ordinal(df_in):
    df_out = df_in.copy()
    # column-specific non-substantive values
    df_out["WB14"] = df_out["WB14"].replace("NO SENTENCE IN REQUIRED LANGUAGE / BRAILLE", np.nan)
    df_out["VT20"] = df_out["VT20"].replace("NEVER WALK ALONE AFTER DARK", np.nan)
    df_out["VT21"] = df_out["VT21"].replace("NEVER WALK ALONE AFTER DARK", np.nan)
    # merge rare categories
    df_out["CB5A"] = df_out["CB5A"].replace({"HIGHER": "UPPER SECONDARY"})
    df_out["WB6A"] = df_out["WB6A"].replace({"ECE": "PRIMARY","VOCATIONAL TRAINING": "UPPER SECONDARY"})
    df_out["AF11"] = df_out["AF11"].replace({"A LOT OF DIFFICULTY": "SOME DIFFICULTY"})
    df_out["AF12"] = df_out["AF12"].replace({"A LOT OF DIFFICULTY": "SOME DIFFICULTY"})
    # simplify grade labels
    grade_map = {"CLASS/YEAR/GRADE 1": "GRADE 1","CLASS/YEAR/GRADE 2": "GRADE 2","CLASS/YEAR/GRADE 3": "GRADE 3",
                 "CLASS/YEAR/GRADE 4": "GRADE 4","CLASS/YEAR/GRADE 5": "GRADE 5","CLASS/GRADE 6": "GRADE 6","CLASS/GRADE 7": "GRADE 7","CLASS/GRADE 8": "GRADE 8"}
    df_out["CB5B"] = df_out["CB5B"].replace(grade_map)
    df_out["WB6B"] = df_out["WB6B"].replace(grade_map)
    return df_out[ordinal_cols]
def ordinal_change_table(df_before, df_after, cols):
    rows = []
    for col in cols:
        if col in df_before.columns:
            before_counts = df_before[col].astype("object").where(df_before[col].notna(), "NaN").value_counts(dropna=False)
            after_counts = df_after[col].astype("object").where(df_after[col].notna(), "NaN").value_counts(dropna=False)
            before_text = " | ".join([f"{k} ({v})" for k, v in before_counts.items()])
            after_text = " | ".join([f"{k} ({v})" for k, v in after_counts.items()])
            n_before = df_before[col].nunique(dropna=True)
            n_after = df_after[col].nunique(dropna=True)
            if (n_before != n_after) or (before_text != after_text):
                rows.append({"column": col, "n_before": n_before, "n_after": n_after, "after": after_text})
    out = pd.DataFrame(rows)
    pd.set_option("display.max_colwidth", None)
    display(out)
    return None
X_train_ord = clean_merge_ordinal(X_train)
ordinal_change_table(X_train[ordinal_cols], X_train_ord[ordinal_cols], ordinal_cols)

,column,n_before,n_after,after
0,CB5A,5,4,PRIMARY (7534) | NaN (566) | ECE (422) | LOWER SECONDARY (242) | UPPER SECONDARY (96)
1,CB5B,8,8,GRADE 1 (1936) | GRADE 2 (1458) | GRADE 3 (1279) | NaN (988) | GRADE 4 (927) | GRADE 5 (809) | GRADE 6 (629) | GRADE 7 (488) | GRADE 8 (346)
2,WB6A,6,4,PRIMARY (6030) | NaN (1048) | LOWER SECONDARY (882) | UPPER SECONDARY (758) | HIGHER (142)
3,WB6B,8,8,GRADE 8 (1342) | GRADE 4 (1260) | GRADE 2 (1136) | NaN (1053) | GRADE 7 (971) | GRADE 3 (912) | GRADE 5 (866) | GRADE 6 (767) | GRADE 1 (553)
4,WB14,4,3,ABLE TO READ WHOLE SENTENCE (3244) | CANNOT READ AT ALL (3063) | NaN (1799) | ABLE TO READ ONLY PARTS OF SENTENCE (754)
5,AF11,3,2,NO DIFFICULTY (8702) | SOME DIFFICULTY (152) | NaN (6)
6,AF12,3,2,NO DIFFICULTY (8647) | SOME DIFFICULTY (206) | NaN (7)
7,VT20,5,4,NaN (3627) | SAFE (2356) | VERY SAFE (1281) | UNSAFE (1192) | VERY UNSAFE (404)
8,VT21,5,4,SAFE (4373) | VERY SAFE (2741) | UNSAFE (1118) | NaN (404) | VERY UNSAFE (224)


The ordinal variables were cleaned and simplified before modelling. Non-substantive responses such as special non-applicable categories were converted to missing values, because they do not represent valid positions on an ordinal scale. In addition, several rare or overly fine-grained categories were merged into nearby levels with similar meaning, such as education levels, grade labels, and difficulty categories. This helps preserve the natural order of the variables, reduces unnecessary sparsity, and produces a more stable representation for the preprocessing pipeline.

In [7]:
# 7. clean and merge nominal categorical variables
nominal_candidates = ["WS1", "WS11", "HC5", "HC4", "ethnicity", "WS3"]
nominal_cols = [col for col in nominal_candidates if col in X_train.columns]
def clean_merge_nominal(df_in):
    df_out = df_in.copy()
    df_out["WS1"] = df_out["WS1"].replace({"PIPED WATER: PIPED INTO DWELLING": "PIPED WATER", "PIPED WATER: PIPED TO YARD / PLOT": "PIPED WATER", "PIPED WATER: PIPED TO NEIGHBOUR": "PIPED WATER",
                                            "PIPED WATER: PUBLIC TAP / STANDPIPE": "PIPED WATER", "TUBE WELL / BOREHOLE": "GROUNDWATER", "DUG WELL: PROTECTED WELL": "GROUNDWATER", 
                                            "DUG WELL: UNPROTECTED WELL": "GROUNDWATER", "SPRING: PROTECTED SPRING": "GROUNDWATER", "SPRING: UNPROTECTED SPRING": "GROUNDWATER", 
                                            "SURFACE WATER (RIVER, DAM, LAKE, POND, STREAM, CANAL, IRRIGATION CHANNEL)": "SURFACE / DELIVERED WATER", "CART WITH SMALL TANK": "SURFACE / DELIVERED WATER", 
                                            "TANKER-TRUCK": "SURFACE / DELIVERED WATER", "PACKAGED WATER: BOTTLED WATER": "PURCHASED WATER", "WATER KIOSK": "PURCHASED WATER"})
    df_out["WS1"] = df_out["WS1"].replace({"PURCHASED WATER": "OTHER WATER", "SURFACE / DELIVERED WATER": "OTHER WATER", "RAINWATER": "OTHER WATER", "OTHER": "OTHER WATER"})
    df_out["WS11"] = df_out["WS11"].replace({"FLUSH / POUR FLUSH: FLUSH TO PIPED SEWER SYSTEM": "FLUSH TOILET", "FLUSH / POUR FLUSH: FLUSH TO SEPTIC TANK": "FLUSH TOILET", 
                                             "FLUSH / POUR FLUSH: FLUSH TO PIT LATRINE": "FLUSH TOILET", "FLUSH / POUR FLUSH: FLUSH TO OPEN DRAIN": "FLUSH TOILET", 
                                             "FLUSH / POUR FLUSH: FLUSH TO DK WHERE": "FLUSH TOILET", "PIT LATRINE: VENTILATED IMPROVED PIT LATRINE": "IMPROVED PIT LATRINE", 
                                             "PIT LATRINE: PIT LATRINE WITH SLAB": "IMPROVED PIT LATRINE", "PIT LATRINE: PIT LATRINE WITHOUT SLAB / OPEN PIT": "UNIMPROVED PIT LATRINE", 
                                             "HANGING TOILET / HANGING LATRINE": "UNIMPROVED PIT LATRINE", "NO FACILITY / BUSH / FIELD": "NO FACILITY", "COMPOSTING TOILET": "OTHER SANITATION", "OTHER": "OTHER SANITATION"})
    df_out["WS11"] = df_out["WS11"].replace({"OTHER SANITATION": "UNIMPROVED PIT LATRINE"})
    df_out["HC5"] = df_out["HC5"].replace({"NO ROOF": "RUDIMENTARY ROOF", "THATCH / PALM LEAF": "RUDIMENTARY ROOF", "RUSTIC MAT": "RUDIMENTARY ROOF", "PALM / BAMBOO": "RUDIMENTARY ROOF", 
                                           "WOOD": "WOOD ROOF", "WOOD PLANKS": "WOOD ROOF", "IRON SHEETS / METAL / TIN": "METAL SHEET ROOF", "CALAMINE / CEMENT FIBRE": "METAL SHEET ROOF", 
                                           "CERAMIC TILES": "FINISHED / DURABLE ROOF", "CEMENT": "FINISHED / DURABLE ROOF", "ROOFING SHINGLES": "FINISHED / DURABLE ROOF"})
    df_out["HC5"] = df_out["HC5"].replace({"WOOD ROOF": "RUDIMENTARY ROOF", "OTHER": "RUDIMENTARY ROOF", "METAL SHEET ROOF": "IMPROVED ROOF", "FINISHED / DURABLE ROOF": "IMPROVED ROOF"})
    df_out["HC4"] = df_out["HC4"].replace({"EARTH / SAND": "RUDIMENTARY FLOOR", "DUNG": "RUDIMENTARY FLOOR", "WOOD PLANKS": "WOOD / NATURAL FLOOR", "PALM / BAMBOO": "WOOD / NATURAL FLOOR", 
                                           "PARQUET OR POLISHED WOOD": "FINISHED FLOOR", "VINYL OR ASPHALT STRIPS": "FINISHED FLOOR", "CERAMIC TILES": "FINISHED FLOOR", "CEMENT": "FINISHED FLOOR", "CARPET": "FINISHED FLOOR"})
    df_out["HC4"] = df_out["HC4"].replace({"WOOD / NATURAL FLOOR": "RUDIMENTARY FLOOR", "OTHER": "RUDIMENTARY FLOOR"})
    df_out["ethnicity"] = df_out["ethnicity"].replace({"Nkhonde": "Other ethnicity"})
    df_out["WS3"] = df_out["WS3"].replace({"IN OWN DWELLING": "ON PREMISES", "IN OWN YARD / PLOT": "ON PREMISES"})
    return df_out[nominal_cols]
def nominal_change_table(df_before, df_after, cols):
    rows = []
    for col in cols:
        before_counts = df_before[col].astype("object").where(df_before[col].notna(), "NaN").value_counts(dropna=False)
        after_counts = df_after[col].astype("object").where(df_after[col].notna(), "NaN").value_counts(dropna=False)
        before_text = " | ".join([f"{k} ({v})" for k, v in before_counts.items()])
        after_text = " | ".join([f"{k} ({v})" for k, v in after_counts.items()])
        n_before = df_before[col].nunique(dropna=True)
        n_after = df_after[col].nunique(dropna=True)
        if (n_before != n_after) or (before_text != after_text): rows.append({"column": col, "n_before": n_before, "n_after": n_after, "after": after_text})
    out = pd.DataFrame(rows)
    pd.set_option("display.max_colwidth", None)
    display(out)
    return None
X_train_nom = clean_merge_nominal(X_train)
nominal_change_table(X_train[nominal_cols], X_train_nom[nominal_cols], nominal_cols)

,column,n_before,n_after,after
0,WS1,15,3,GROUNDWATER (6808) | PIPED WATER (1708) | OTHER WATER (344)
1,WS11,12,4,IMPROVED PIT LATRINE (6819) | UNIMPROVED PIT LATRINE (1195) | NO FACILITY (647) | FLUSH TOILET (197) | NaN (2)
2,HC5,12,2,IMPROVED ROOF (5020) | RUDIMENTARY ROOF (3840)
3,HC4,10,2,RUDIMENTARY FLOOR (6313) | FINISHED FLOOR (2547)
4,ethnicity,9,8,Chewa (2693) | Lomwe (1616) | Ngoni (1055) | Yao (990) | Other ethnicity (904) | Tumbuka (853) | Sena (444) | Tonga (305)
5,WS3,3,2,ELSEWHERE (7700) | NaN (779) | ON PREMISES (381)


For nominal categorical variables, categories were merged according to substantive meaning rather than order. This was mainly done to reduce unnecessary sparsity, combine extremely small groups, and preserve the broader social meaning of variables such as water source, sanitation type, housing quality, ethnicity, and water access location.

In [8]:
# 8. apply ordinal and nominal cleaning to all splits
X_train[ordinal_cols] = clean_merge_ordinal(X_train)
X_val[ordinal_cols] = clean_merge_ordinal(X_val)
X_test[ordinal_cols] = clean_merge_ordinal(X_test)
X_train[nominal_cols] = clean_merge_nominal(X_train)
X_val[nominal_cols] = clean_merge_nominal(X_val)
X_test[nominal_cols] = clean_merge_nominal(X_test)

In [9]:
# 9. define final variable types
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()
ordinal_cols = [col for col in ordinal_candidates if col in cat_cols]
nominal_cols = [col for col in cat_cols if col not in ordinal_cols]
print("num of num_cols:", len(num_cols),"; num of ordinal_cols:",len(ordinal_cols),"; num of nominal_cols:",len(nominal_cols))

num of num_cols: 9 ; num of ordinal_cols: 13 ; num of nominal_cols: 59


In [10]:
# 10. define ordinal categories
ordinal_categories = [
    ["ECE", "PRIMARY", "LOWER SECONDARY", "UPPER SECONDARY"],
    ["GRADE 1", "GRADE 2", "GRADE 3", "GRADE 4", "GRADE 5", "GRADE 6", "GRADE 7", "GRADE 8"],
    ["PRIMARY", "LOWER SECONDARY", "UPPER SECONDARY", "HIGHER"],
    ["GRADE 1", "GRADE 2", "GRADE 3", "GRADE 4", "GRADE 5", "GRADE 6", "GRADE 7", "GRADE 8"],
    ["CANNOT READ AT ALL", "ABLE TO READ ONLY PARTS OF SENTENCE", "ABLE TO READ WHOLE SENTENCE"],
    ["NO DIFFICULTY", "SOME DIFFICULTY", "A LOT OF DIFFICULTY"],
    ["NO DIFFICULTY", "SOME DIFFICULTY"],
    ["NO DIFFICULTY", "SOME DIFFICULTY"],
    ["VERY UNHAPPY", "SOMEWHAT UNHAPPY", "NEITHER HAPPY NOR UNHAPPY", "SOMEWHAT HAPPY", "VERY HAPPY"],
    ["WORSENED", "MORE OR LESS THE SAME", "IMPROVED"],
    ["WORSE", "MORE OR LESS THE SAME", "BETTER"],
    ["VERY UNSAFE", "UNSAFE", "SAFE", "VERY SAFE"],
    ["VERY UNSAFE", "UNSAFE", "SAFE", "VERY SAFE"]]

In [11]:
# 11. compact correlation summary table
top_pos = corr_with_y.sort_values(ascending=False).head(10).reset_index().rename(columns={"index": "feature", 0: "corr"})
top_neg = corr_with_y.sort_values().head(10).reset_index().rename(columns={"index": "feature", 0: "corr"})
corr_summary = pd.DataFrame({"top_positive_feature": top_pos["feature"],"top_positive_corr": top_pos["corr"].round(4),"top_negative_feature": top_neg["feature"],"top_negative_corr": top_neg["corr"].round(4)}).T
corr_summary.columns = [f"{i+1}" for i in range(corr_summary.shape[1])]
display(corr_summary.iloc[:, :3])

NameError: name 'corr_with_y' is not defined

In [ ]:
# 12. count features with very low correlation to label
low_corr_count = (corr_with_y.abs() < 0.005).sum()
print("num of features with |corr| < 0.005:", low_corr_count)

num of features with |corr| < 0.005: 14


Pearson correlation was used as a preliminary filter to identify encoded features with almost no linear association with the binary target [3]. It can be seen that the most relevant correlation coefficient is only 0.1. Therefore, I followed the calculation method of 0.1*0.05=0.005 to reduce features as much as possible without disrupting the data structure.

[3] Ge, G., & Zhang, J. (2023). Feature selection methods and predictive models in CT lung cancer radiomics. Journal of applied clinical medical physics, 24(1), e13869. https://doi.org/10.1002/acm2.13869

In [146]:
# 13. build preprocessing pipeline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])
ordinal_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("encoder", OrdinalEncoder(categories=ordinal_categories))])
nominal_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("encoder", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))])

preprocessor = ColumnTransformer([("num", num_pipe, num_cols),("ord", ordinal_pipe, ordinal_cols),("nom", nominal_pipe, nominal_cols)])

# 14. fit preprocessor and transform all splits
preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)
# remove features with |corr| < 0.005
feature_names = preprocessor.get_feature_names_out()
low_corr_features = corr_with_y[corr_with_y.abs() < 0.005].index.tolist()
keep_idx = [i for i, f in enumerate(feature_names) if f not in low_corr_features]
X_train_processed = X_train_processed[:, keep_idx]
X_val_processed = X_val_processed[:, keep_idx]
X_test_processed = X_test_processed[:, keep_idx]
feature_names = feature_names[keep_idx]

In [147]:
# 15. check processed data
print("X_train_processed:", X_train_processed.shape)
print("X_val_processed:", X_val_processed.shape)
print("X_test_processed:", X_test_processed.shape)
print("num of final features:", len(feature_names))
print("NaN in X_train_processed:", np.isnan(X_train_processed).sum())
print("NaN in X_val_processed:", np.isnan(X_val_processed).sum())
print("NaN in X_test_processed:", np.isnan(X_test_processed).sum())

X_train_processed: (8860, 96)
X_val_processed: (1880, 96)
X_test_processed: (1879, 96)
num of final features: 96
NaN in X_train_processed: 0
NaN in X_val_processed: 0
NaN in X_test_processed: 0


# Model Fitting and Tuning

*In this section you should detail and motivate your choice of model and describe the process used to refine, tune, and fit that model. You are encouraged to explore different models but you should NOT include a detailed narrative or code of all of these attempts. At most this section should very briefly mention the methods explored and why they were rejected - most of your effort should go into describing the final model you are using and your process for tuning and validating it.*

*This section should include the full implementation of your final model, including all necessary validation. As with figures, any included code must also be addressed in the text of the document.*

*You should also include a baseline model of your choice and provide a comparison of your model with the baseline model on the test data. You should briefly describe the baseline model considered.*

### RF

In [130]:
# Random Forest on raw data vs engineered data

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score
import pandas as pd
import numpy as np

def evaluate_binary(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, proba),
        "avg_precision": average_precision_score(y_true, proba),
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred)
    }

rf_grid = [
    {"n_estimators": 200, "max_depth": 8, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 10, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 12, "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 400, "max_depth": None, "min_samples_leaf": 1, "max_features": "sqrt"}
]

# --------------------------------------------------
# 1. processed data
# --------------------------------------------------
proc_results = []
for params in rf_grid:
    rf = RandomForestClassifier(random_state=11205, n_jobs=-1, **params)
    rf.fit(X_train_processed, y_train.astype(int))
    val_proba = rf.predict_proba(X_val_processed)[:, 1]
    row = {"data_version": "processed", **params}
    row.update(evaluate_binary(y_val.astype(int), val_proba))
    proc_results.append(row)

proc_val_df = pd.DataFrame(proc_results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
best_proc_params = {k: proc_val_df.loc[0, k] for k in ["n_estimators", "max_depth", "min_samples_leaf", "max_features"]}
best_proc_params["n_estimators"] = int(best_proc_params["n_estimators"])
best_proc_params["max_depth"] = None if pd.isna(best_proc_params["max_depth"]) else int(best_proc_params["max_depth"])
best_proc_params["min_samples_leaf"] = int(best_proc_params["min_samples_leaf"])

X_trainval_processed = np.vstack([X_train_processed, X_val_processed])
y_trainval_processed = np.concatenate([y_train.astype(int), y_val.astype(int)])

best_proc_rf = RandomForestClassifier(random_state=11205, n_jobs=-1, **best_proc_params)
best_proc_rf.fit(X_trainval_processed, y_trainval_processed)
proc_test_proba = best_proc_rf.predict_proba(X_test_processed)[:, 1]
proc_test_metrics = evaluate_binary(y_test.astype(int), proc_test_proba)

# --------------------------------------------------
# 2. raw data
# --------------------------------------------------
df_raw = df.copy()
df_raw = df_raw[df_raw["FCF26"].notna() & (df_raw["FCF26"] != "NO RESPONSE")].copy()
df_raw["target"] = (df_raw["FCF26"] != "NEVER").astype(int)
X_raw = df_raw.drop(columns=["FCF26", "target", "HH1", "HH2", "LN", "FS4"]).copy()
y_raw = df_raw["target"].copy()

X_train_raw, X_temp_raw, y_train_raw, y_temp_raw = train_test_split(
    X_raw, y_raw, test_size=0.30, random_state=11205, stratify=y_raw
)
X_val_raw, X_test_raw, y_val_raw, y_test_raw = train_test_split(
    X_temp_raw, y_temp_raw, test_size=0.50, random_state=11205, stratify=y_temp_raw
)

num_cols_raw = X_train_raw.select_dtypes(include=np.number).columns.tolist()
cat_cols_raw = X_train_raw.select_dtypes(exclude=np.number).columns.tolist()

num_pipe_raw = Pipeline([("imputer", SimpleImputer(strategy="median"))])
cat_pipe_raw = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
])

raw_preprocessor = ColumnTransformer([
    ("num", num_pipe_raw, num_cols_raw),
    ("cat", cat_pipe_raw, cat_cols_raw)
])

X_train_raw_processed = raw_preprocessor.fit_transform(X_train_raw)
X_val_raw_processed = raw_preprocessor.transform(X_val_raw)
X_test_raw_processed = raw_preprocessor.transform(X_test_raw)

raw_results = []
for params in rf_grid:
    rf = RandomForestClassifier(random_state=11205, n_jobs=-1, **params)
    rf.fit(X_train_raw_processed, y_train_raw.astype(int))
    val_proba = rf.predict_proba(X_val_raw_processed)[:, 1]
    row = {"data_version": "raw", **params}
    row.update(evaluate_binary(y_val_raw.astype(int), val_proba))
    raw_results.append(row)

raw_val_df = pd.DataFrame(raw_results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
best_raw_params = {k: raw_val_df.loc[0, k] for k in ["n_estimators", "max_depth", "min_samples_leaf", "max_features"]}
best_raw_params["n_estimators"] = int(best_raw_params["n_estimators"])
best_raw_params["max_depth"] = None if pd.isna(best_raw_params["max_depth"]) else int(best_raw_params["max_depth"])
best_raw_params["min_samples_leaf"] = int(best_raw_params["min_samples_leaf"])

X_trainval_raw_processed = np.vstack([X_train_raw_processed, X_val_raw_processed])
y_trainval_raw = np.concatenate([y_train_raw.astype(int), y_val_raw.astype(int)])

best_raw_rf = RandomForestClassifier(random_state=11205, n_jobs=-1, **best_raw_params)
best_raw_rf.fit(X_trainval_raw_processed, y_trainval_raw)
raw_test_proba = best_raw_rf.predict_proba(X_test_raw_processed)[:, 1]
raw_test_metrics = evaluate_binary(y_test_raw.astype(int), raw_test_proba)

# --------------------------------------------------
# 3. final comparison
# --------------------------------------------------
summary_df = pd.DataFrame([
    {"data_version": "processed", "best_params": str(best_proc_params), **proc_test_metrics},
    {"data_version": "raw", "best_params": str(best_raw_params), **raw_test_metrics}
])

display(summary_df)

d:\Anaconda\envs\scanpy_env\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [5, 42, 63, 70] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\Anaconda\envs\scanpy_env\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1, 5, 35, 42, 68, 70] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,data_version,best_params,roc_auc,avg_precision,accuracy,f1
0,processed,"{'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 1, 'max_features': 'sqrt'}",0.633745,0.659249,0.604045,0.682051
1,raw,"{'n_estimators': 400, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt'}",0.637349,0.666025,0.604806,0.679917


In [131]:
# RF under three low-correlation thresholds: 0.001, 0.005, 0.01

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score
import pandas as pd
import numpy as np

def evaluate_binary(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, proba),
        "avg_precision": average_precision_score(y_true, proba),
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred)
    }

rf_grid = [
    {"n_estimators": 200, "max_depth": 8, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 10, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 12, "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 400, "max_depth": None, "min_samples_leaf": 1, "max_features": "sqrt"}
]

y_train_int = y_train.astype(int)
y_val_int = y_val.astype(int)
y_test_int = y_test.astype(int)

corr_processed = pd.Series(
    [np.corrcoef(X_train_processed[:, i], y_train_int)[0, 1] for i in range(X_train_processed.shape[1])],
    index=feature_names
).fillna(0.0)

threshold_list = [0.001, 0.005, 0.01]
val_rows, test_rows = [], []

for th in threshold_list:
    keep_mask = corr_processed.abs() >= th
    keep_idx = np.where(keep_mask)[0]
    kept_features = corr_processed.index[keep_mask]

    X_train_th = X_train_processed[:, keep_idx]
    X_val_th = X_val_processed[:, keep_idx]
    X_test_th = X_test_processed[:, keep_idx]

    th_results = []
    for params in rf_grid:
        rf = RandomForestClassifier(random_state=11205, n_jobs=-1, **params)
        rf.fit(X_train_th, y_train_int)
        val_proba = rf.predict_proba(X_val_th)[:, 1]
        row = {"corr_threshold": th, "n_features": X_train_th.shape[1], **params}
        row.update(evaluate_binary(y_val_int, val_proba))
        th_results.append(row)

    th_val_df = pd.DataFrame(th_results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
    best_row = th_val_df.loc[0]

    best_params = {
        "n_estimators": int(best_row["n_estimators"]),
        "max_depth": None if pd.isna(best_row["max_depth"]) else (None if str(best_row["max_depth"]) == "None" else int(best_row["max_depth"])),
        "min_samples_leaf": int(best_row["min_samples_leaf"]),
        "max_features": best_row["max_features"]
    }

    X_trainval_th = np.vstack([X_train_th, X_val_th])
    y_trainval_th = np.concatenate([y_train_int, y_val_int])

    best_rf = RandomForestClassifier(random_state=11205, n_jobs=-1, **best_params)
    best_rf.fit(X_trainval_th, y_trainval_th)
    test_proba = best_rf.predict_proba(X_test_th)[:, 1]
    test_metrics = evaluate_binary(y_test_int, test_proba)

    val_rows.append({
        "corr_threshold": th,
        "n_features": X_train_th.shape[1],
        "features_removed": X_train_processed.shape[1] - X_train_th.shape[1],
        "best_params": str(best_params),
        "val_roc_auc": round(best_row["roc_auc"], 4),
        "val_avg_precision": round(best_row["avg_precision"], 4),
        "val_accuracy": round(best_row["accuracy"], 4),
        "val_f1": round(best_row["f1"], 4)
    })

    test_rows.append({
        "corr_threshold": th,
        "n_features": X_train_th.shape[1],
        "features_removed": X_train_processed.shape[1] - X_train_th.shape[1],
        "best_params": str(best_params),
        "test_roc_auc": round(test_metrics["roc_auc"], 4),
        "test_avg_precision": round(test_metrics["avg_precision"], 4),
        "test_accuracy": round(test_metrics["accuracy"], 4),
        "test_f1": round(test_metrics["f1"], 4)
    })

val_summary = pd.DataFrame(val_rows)
test_summary = pd.DataFrame(test_rows)

display(val_summary)
display(test_summary)

,corr_threshold,n_features,features_removed,best_params,val_roc_auc,val_avg_precision,val_accuracy,val_f1
0,0.001,94,2,"{'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 1, 'max_features': 'sqrt'}",0.6516,0.6914,0.6117,0.6872
1,0.005,88,8,"{'n_estimators': 400, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt'}",0.6539,0.6866,0.6149,0.6811
2,0.010,81,15,"{'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 1, 'max_features': 'sqrt'}",0.6500,0.6883,0.6090,0.6822


,corr_threshold,n_features,features_removed,best_params,test_roc_auc,test_avg_precision,test_accuracy,test_f1
0,0.001,94,2,"{'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 1, 'max_features': 'sqrt'}",0.6345,0.6600,0.6040,0.6804
1,0.005,88,8,"{'n_estimators': 400, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt'}",0.6343,0.6579,0.6062,0.6771
2,0.010,81,15,"{'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 1, 'max_features': 'sqrt'}",0.6347,0.6634,0.6003,0.6748


In [132]:
# RF after removing very low-correlation features (|corr| < 0.005) and applying PCA

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score
import pandas as pd
import numpy as np

def evaluate_binary(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, proba),
        "avg_precision": average_precision_score(y_true, proba),
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred)
    }

# 1. remove features with |corr| < 0.005
low_corr_features = corr_with_y[corr_with_y.abs() < 0.005].index.tolist()
keep_idx = [i for i, f in enumerate(feature_names) if f not in low_corr_features]

X_train_rf = X_train_processed[:, keep_idx]
X_val_rf = X_val_processed[:, keep_idx]
X_test_rf = X_test_processed[:, keep_idx]
feature_names_rf = feature_names[keep_idx]

# 2. PCA grid
pca_grid = [0.80, 0.90, 0.95, 0.99]

# 3. RF grid
rf_grid = [
    {"n_estimators": 200, "max_depth": 8, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 10, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 12, "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 400, "max_depth": None, "min_samples_leaf": 1, "max_features": "sqrt"}
]

y_train_int = y_train.astype(int)
y_val_int = y_val.astype(int)
y_test_int = y_test.astype(int)

# 4. validation search
val_results = []
for pca_keep in pca_grid:
    pca = PCA(n_components=pca_keep, random_state=11205)
    X_train_pca = pca.fit_transform(X_train_rf)
    X_val_pca = pca.transform(X_val_rf)

    for params in rf_grid:
        rf = RandomForestClassifier(random_state=11205, n_jobs=-1, **params)
        rf.fit(X_train_pca, y_train_int)
        val_proba = rf.predict_proba(X_val_pca)[:, 1]

        row = {
            "corr_threshold": 0.005,
            "n_features_before_pca": X_train_rf.shape[1],
            "pca_n_components": pca.n_components_,
            "pca_variance_kept": pca_keep,
            **params
        }
        row.update(evaluate_binary(y_val_int, val_proba))
        val_results.append(row)

val_df = pd.DataFrame(val_results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
best_row = val_df.loc[0]

best_params = {
    "n_estimators": int(best_row["n_estimators"]),
    "max_depth": None if pd.isna(best_row["max_depth"]) else (None if str(best_row["max_depth"]) == "None" else int(best_row["max_depth"])),
    "min_samples_leaf": int(best_row["min_samples_leaf"]),
    "max_features": best_row["max_features"]
}
best_pca_keep = float(best_row["pca_variance_kept"])

# 5. refit on train + val, evaluate on test
X_trainval_rf = np.vstack([X_train_rf, X_val_rf])
y_trainval_rf = np.concatenate([y_train_int, y_val_int])

best_pca = PCA(n_components=best_pca_keep, random_state=11205)
X_trainval_pca = best_pca.fit_transform(X_trainval_rf)
X_test_pca = best_pca.transform(X_test_rf)

best_rf = RandomForestClassifier(random_state=11205, n_jobs=-1, **best_params)
best_rf.fit(X_trainval_pca, y_trainval_rf)
test_proba = best_rf.predict_proba(X_test_pca)[:, 1]
test_metrics = evaluate_binary(y_test_int, test_proba)

# 6. final output
summary_df = pd.DataFrame([{
    "corr_threshold": 0.005,
    "features_removed": len(low_corr_features),
    "n_features_before_pca": X_train_rf.shape[1],
    "best_pca_variance_kept": best_pca_keep,
    "best_pca_n_components": best_pca.n_components_,
    "best_params": str(best_params),
    **test_metrics
}])

display(summary_df)

,corr_threshold,features_removed,n_features_before_pca,best_pca_variance_kept,best_pca_n_components,best_params,roc_auc,avg_precision,accuracy,f1
0,0.005,14,96,0.99,78,"{'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 1, 'max_features': 'sqrt'}",0.615742,0.639811,0.603513,0.690229


In [133]:
# Dimension reduction for numeric / ordinal / nominal features + compare RF and LR

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score
import pandas as pd
import numpy as np

def evaluate_binary(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, proba),
        "avg_precision": average_precision_score(y_true, proba),
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred)
    }

def drop_high_corr_cols(df_in, threshold=0.90):
    corr = df_in.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_cols = [col for col in upper.columns if (upper[col] > threshold).any()]
    return df_in.drop(columns=drop_cols), drop_cols

# --------------------------------------------------
# 1. numeric + ordinal block
# --------------------------------------------------
num_imputer = SimpleImputer(strategy="median")
ord_imputer = SimpleImputer(strategy="most_frequent")
ord_encoder = OrdinalEncoder(categories=ordinal_categories)

X_train_num = pd.DataFrame(num_imputer.fit_transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
X_val_num = pd.DataFrame(num_imputer.transform(X_val[num_cols]), columns=num_cols, index=X_val.index)
X_test_num = pd.DataFrame(num_imputer.transform(X_test[num_cols]), columns=num_cols, index=X_test.index)

X_train_ord = pd.DataFrame(ord_encoder.fit_transform(ord_imputer.fit_transform(X_train[ordinal_cols])), columns=ordinal_cols, index=X_train.index)
X_val_ord = pd.DataFrame(ord_encoder.transform(ord_imputer.transform(X_val[ordinal_cols])), columns=ordinal_cols, index=X_val.index)
X_test_ord = pd.DataFrame(ord_encoder.transform(ord_imputer.transform(X_test[ordinal_cols])), columns=ordinal_cols, index=X_test.index)

X_train_numord = pd.concat([X_train_num, X_train_ord], axis=1)
X_val_numord = pd.concat([X_val_num, X_val_ord], axis=1)
X_test_numord = pd.concat([X_test_num, X_test_ord], axis=1)

X_train_numord_red, numord_drop_cols = drop_high_corr_cols(X_train_numord, threshold=0.90)
keep_numord_cols = X_train_numord_red.columns.tolist()
X_val_numord_red = X_val_numord[keep_numord_cols].copy()
X_test_numord_red = X_test_numord[keep_numord_cols].copy()

# --------------------------------------------------
# 2. nominal block
# --------------------------------------------------
nom_imputer = SimpleImputer(strategy="most_frequent")
nom_encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)

X_train_nom_imp = nom_imputer.fit_transform(X_train[nominal_cols])
X_val_nom_imp = nom_imputer.transform(X_val[nominal_cols])
X_test_nom_imp = nom_imputer.transform(X_test[nominal_cols])

X_train_nom_ohe = nom_encoder.fit_transform(X_train_nom_imp)
X_val_nom_ohe = nom_encoder.transform(X_val_nom_imp)
X_test_nom_ohe = nom_encoder.transform(X_test_nom_imp)

nom_feature_names = nom_encoder.get_feature_names_out(nominal_cols)

var_filter = VarianceThreshold(threshold=0.001)
X_train_nom_var = var_filter.fit_transform(X_train_nom_ohe)
X_val_nom_var = var_filter.transform(X_val_nom_ohe)
X_test_nom_var = var_filter.transform(X_test_nom_ohe)
nom_var_names = nom_feature_names[var_filter.get_support()]

mi_scores = pd.Series(mutual_info_classif(X_train_nom_var, y_train.astype(int), random_state=11205), index=nom_var_names)
mi_keep_names = mi_scores[mi_scores > 0].index.tolist()
mi_keep_idx = [i for i, f in enumerate(nom_var_names) if f in mi_keep_names]

X_train_nom_red = X_train_nom_var[:, mi_keep_idx]
X_val_nom_red = X_val_nom_var[:, mi_keep_idx]
X_test_nom_red = X_test_nom_var[:, mi_keep_idx]
nom_final_names = nom_var_names[mi_keep_idx]

# --------------------------------------------------
# 3. final reduced matrices
# --------------------------------------------------
X_train_red = np.hstack([X_train_numord_red.to_numpy(), X_train_nom_red])
X_val_red = np.hstack([X_val_numord_red.to_numpy(), X_val_nom_red])
X_test_red = np.hstack([X_test_numord_red.to_numpy(), X_test_nom_red])

final_feature_names = np.array(keep_numord_cols + list(nom_final_names))

# --------------------------------------------------
# 4. Random Forest grid
# --------------------------------------------------
rf_grid = [
    {"n_estimators": 200, "max_depth": 8, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 10, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 12, "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 400, "max_depth": None, "min_samples_leaf": 1, "max_features": "sqrt"}
]

rf_rows = []
for params in rf_grid:
    rf = RandomForestClassifier(random_state=11205, n_jobs=-1, **params)
    rf.fit(X_train_red, y_train.astype(int))
    val_proba = rf.predict_proba(X_val_red)[:, 1]
    row = {"model": "RandomForest", **params}
    row.update(evaluate_binary(y_val.astype(int), val_proba))
    rf_rows.append(row)

rf_val_df = pd.DataFrame(rf_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
best_rf_row = rf_val_df.loc[0]
best_rf_params = {
    "n_estimators": int(best_rf_row["n_estimators"]),
    "max_depth": None if pd.isna(best_rf_row["max_depth"]) else (None if str(best_rf_row["max_depth"]) == "None" else int(best_rf_row["max_depth"])),
    "min_samples_leaf": int(best_rf_row["min_samples_leaf"]),
    "max_features": best_rf_row["max_features"]
}

X_trainval_red = np.vstack([X_train_red, X_val_red])
y_trainval = np.concatenate([y_train.astype(int), y_val.astype(int)])

best_rf = RandomForestClassifier(random_state=11205, n_jobs=-1, **best_rf_params)
best_rf.fit(X_trainval_red, y_trainval)
rf_test_proba = best_rf.predict_proba(X_test_red)[:, 1]
rf_test_metrics = evaluate_binary(y_test.astype(int), rf_test_proba)

# --------------------------------------------------
# 5. Logistic Regression grid
# --------------------------------------------------
scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train_red)
X_val_lr = scaler.transform(X_val_red)
X_test_lr = scaler.transform(X_test_red)
X_trainval_lr = scaler.fit_transform(X_trainval_red)
X_test_lr_final = scaler.transform(X_test_red)

lr_grid = [0.001, 0.01, 0.1, 1, 10, 100]

lr_rows = []
for C in lr_grid:
    lr = LogisticRegression(C=C, solver="liblinear", max_iter=1000, random_state=11205)
    lr.fit(X_train_lr, y_train.astype(int))
    val_proba = lr.predict_proba(X_val_lr)[:, 1]
    row = {"model": "LogisticRegression", "C": C}
    row.update(evaluate_binary(y_val.astype(int), val_proba))
    lr_rows.append(row)

lr_val_df = pd.DataFrame(lr_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
best_C = float(lr_val_df.loc[0, "C"])

best_lr = LogisticRegression(C=best_C, solver="liblinear", max_iter=1000, random_state=11205)
best_lr.fit(X_trainval_lr, y_trainval)
lr_test_proba = best_lr.predict_proba(X_test_lr_final)[:, 1]
lr_test_metrics = evaluate_binary(y_test.astype(int), lr_test_proba)

# --------------------------------------------------
# 6. final comparison
# --------------------------------------------------
summary_df = pd.DataFrame([
    {
        "model": "RandomForest",
        "best_setting": str(best_rf_params),
        "num_features_final": X_train_red.shape[1],
        "num_numord_dropped": len(numord_drop_cols),
        "num_nominal_dropped_by_variance_or_mi": len(nom_feature_names) - len(nom_final_names),
        **rf_test_metrics
    },
    {
        "model": "LogisticRegression",
        "best_setting": f"C={best_C}",
        "num_features_final": X_train_red.shape[1],
        "num_numord_dropped": len(numord_drop_cols),
        "num_nominal_dropped_by_variance_or_mi": len(nom_feature_names) - len(nom_final_names),
        **lr_test_metrics
    }
])

display(summary_df)

,model,best_setting,num_features_final,num_numord_dropped,num_nominal_dropped_by_variance_or_mi,roc_auc,avg_precision,accuracy,f1
0,RandomForest,"{'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 1, 'max_features': 'sqrt'}",64,0,32,0.625381,0.653365,0.596594,0.677172
1,LogisticRegression,C=0.001,64,0,32,0.616153,0.655312,0.593933,0.650160


# Interpretation, Discussion & Conclusions

*In this section you should provide a general overview of your final model, its performance, and reliability. You should discuss what the implications of your model are in terms of the included features, estimated parameters and relationships, predictive performance, and anything else you think is relevant.*

*This should be written with a target audience of a government official, who understands the issues associated with mental health but may only have university level mathematics (not necessarily postgraduate statistics or machine learning). Your goal should be to highlight to this audience how your model can useful. You should also discuss potential limitations or directions of future improvement of your model.*

*Finally, you should include recommendations on factors that may increase the risk of depression, which may be useful for the government officials and health care workers to improve their understanding of the condition, and potentially assit in the development of effective social and health policies and interventions.*

*Keep in mind that a negative result, i.e. a model that does not work well predictively, that is well explained and justified in terms of why it failed will likely receive higher marks than a model with strong predictive performance but with poor or incorrect explanations / justifications.*

# Generative AI statement

*Include a statement on how generative AI was used in the project and report.*

# References

*Include references if any*

In [191]:
import os
os.chdir(r"D:\Ed-Machine Learning in Python\ML-ICA\project_materials\project_materials")
# Run the following to render to PDF
!jupyter nbconvert --to pdf "project - draft3.ipynb"

[NbConvertApp] Converting notebook project - draft3.ipynb to pdf
[NbConvertApp] Writing 195675 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | b had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 146702 bytes to project - draft3.pdf
